# 5.20 — Normalizing Features Using `MinMaxScaler`

**Normalization** rescales numerical features into a bounded range (commonly $[0, 1]$):

$$
 x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

In this repo you can choose the numeric scaling strategy when fitting the preprocessor:

- `numeric_scaler="standard"` → `StandardScaler` (mean 0, std 1)
- `numeric_scaler="minmax"` → `MinMaxScaler` (range $[0, 1]$ on training data)

Leakage rule:

- **Split first**
- **Fit scaler on `X_train` only**
- **Transform `X_test` using the fitted scaler**

In [ ]:
import pandas as pd

from src.data_loader import load_csv
from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_SOURCE_COLUMN,
)
from src.preprocessing import (
    separate_features_and_target,
    split_data,
    fit_preprocessor,
    transform_features,
)

## 1) Load demo data and derive the target

For the demo dataset, `yield_kg` is used only to derive the label and is excluded from features.

In [ ]:
df = load_csv("data/raw/source_demo_crops_20260321.csv")

working = df.copy()
working[TARGET_COLUMN] = (working[TARGET_SOURCE_COLUMN] >= working[TARGET_SOURCE_COLUMN].median()).astype(int)

X, y = separate_features_and_target(
    working,
    target_column=TARGET_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in working.columns],
    excluded_columns=EXCLUDED_COLUMNS,
)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))
X.head()

## 2) Correct workflow: split first, then fit `MinMaxScaler` on train only

`MinMaxScaler` guarantees that (for each numeric feature) the *training* split has:

- min $ 0$
- max $ 1$

Test values may be **outside $[0, 1]$** if they fall outside the training min/max. That is expected.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

bundle = fit_preprocessor(
    X_train,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    numeric_scaler="minmax",
)

X_train_t = transform_features(X_train, bundle)
X_test_t = transform_features(X_test, bundle)

scaled_numeric_columns = [c for c in NUMERICAL_FEATURES if c in X_train_t.columns]

print("Scaled numeric columns:", scaled_numeric_columns)
print("Train mins (should be ~0):")
print(X_train_t[scaled_numeric_columns].min(axis=0))
print("Train maxes (should be ~1):")
print(X_train_t[scaled_numeric_columns].max(axis=0))

print("Test mins / maxes (may be outside [0, 1]):")
print(X_test_t[scaled_numeric_columns].min(axis=0))
print(X_test_t[scaled_numeric_columns].max(axis=0))

## 3) What NOT to do (leakage): fit on full data before splitting

This is intentionally incorrect:

- Fitting the scaler on full `X` uses test-set min/max to transform training, leaking test distribution information.

In [ ]:
# WRONG: fit preprocessor on full dataset (leakage)
leaky_bundle = fit_preprocessor(
    X,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    numeric_scaler="minmax",
)

X_all_t = transform_features(X, leaky_bundle)
X_train_leaky, X_test_leaky, _, _ = split_data(X_all_t, y, test_size=0.2, random_state=42)

scaled_numeric_columns_leaky = [c for c in NUMERICAL_FEATURES if c in X_train_leaky.columns]

print("Leaky train mins (often not exactly 0):")
print(X_train_leaky[scaled_numeric_columns_leaky].min(axis=0))
print("Leaky train maxes (often not exactly 1):")
print(X_train_leaky[scaled_numeric_columns_leaky].max(axis=0))

## 4) Outlier sensitivity (why MinMaxScaler can compress most values)

A single extreme outlier stretches the min/max range and compresses the majority of values near 0.

This is why you should inspect distributions (histograms/boxplots) before choosing normalization.

In [ ]:
values = pd.Series([100, 200, 300, 400, 500, 600, 700, 800, 900, 50_000], name="account_balance")

# Manual min-max scaling for demonstration
vmin = float(values.min())
vmax = float(values.max())
scaled = (values - vmin) / (vmax - vmin)

pd.DataFrame({"original": values, "minmax_scaled": scaled})

## Takeaways

- Use `MinMaxScaler` when you want bounded numeric inputs (e.g., distance-based models, neural nets).
- Do **not** fit normalization on the full dataset.
- Expect test values to sometimes go below 0 or above 1 when the test distribution exceeds training min/max.
- Watch out for outliers: they can compress most values into a tiny interval.